In [5]:
import os
import xarray as xr
import datetime
import pandas as pd

# 1) 기본 경로 설정
input_folder = r'C:\Yeonji\2025.01.Drought\2004\1.CoKriging(Daily)(Clip_nc)'
base_output  = r'C:\2025.01.Drought\2004'

# 2) 날짜별 파일 매핑
file_list = sorted([
    f for f in os.listdir(input_folder)
    if f.endswith('.nc') and '_Cokriging_Precipitation' in f
])
date_file_map = {}
for fname in file_list:
    date_str = fname.split('_')[0]  # 'YYYY.MM.DD'
    try:
        dt = datetime.datetime.strptime(date_str, "%Y%m%d")
        # pandas Timestamp 로 변환해야 그룹핑 편리
        ts = pd.Timestamp(dt.date())
        date_file_map[ts] = os.path.join(input_folder, fname)
    except Exception as e:
        print(f"❌ 날짜 파싱 실패: {fname} → {e}")

# 3) 월별 날짜 그룹핑
all_dates = sorted(date_file_map.keys())
df = pd.DataFrame({'date': all_dates})
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # Ensure datetime type
df['ym'] = df['date'].dt.to_period('M').astype(str)  # 'YYYY-MM'
monthly_dates = df.groupby('ym')['date'].apply(list).to_dict()
sorted_months = sorted(monthly_dates.keys())

# 4) 누적 기간 정의
window_sizes = [3, 6, 9, 12]

for window in window_sizes:
    out_folder = os.path.join(base_output, f'{window}mon_output_Cokriging')
    os.makedirs(out_folder, exist_ok=True)

    # 슬라이딩 윈도우
    for i in range(len(sorted_months) - window + 1):
        months = sorted_months[i:i+window]
        dates = sum((monthly_dates[m] for m in months), [])
        out_yyyymm = months[-1].replace('-', '')  # e.g. '202503'

        ds_list = []
        for date in dates:
            path = date_file_map.get(date)
            if not path or not os.path.exists(path):
                print(f"⚠️ 파일 없음: {date} → {path}")
                continue
            ds = xr.open_dataset(path)
            if 'precipitation' not in ds:
                print(f"⚠️ 변수 없음: {path}")
                ds.close()
                continue
            ds_list.append(ds)

        # 데이터를 충분히 모았는지 체크 (window*20일 정도 기준)
        if len(ds_list) < window * 20:
            print(f"⚠️ 스킵 {out_yyyymm} ({window}개월, 일수 부족: {len(ds_list)})")
            for ds in ds_list: ds.close()
            continue

        # 누적 합 계산
        total = sum(ds['precipitation'] for ds in ds_list)
        result = ds_list[0].copy(deep=True)
        result['precipitation'] = total

        # 저장
        out_path = os.path.join(
            out_folder,
            f"{out_yyyymm}_Cokriging_{window}mon_Precipitation.nc"
        )
        result.to_netcdf(out_path)
        print(f"✅ 저장 ({window}개월): {out_path}")

        # 리소스 해제
        for ds in ds_list: ds.close()
        result.close()


✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200403_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200404_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200405_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200406_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200407_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200408_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200409_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200410_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200411_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought\2004\3mon_output_Cokriging\200412_Cokriging_3mon_Precipitation.nc
✅ 저장 (3개월): C:\2025.01.Drought

KeyboardInterrupt: 

In [8]:
import os
import xarray as xr
import numpy as np
import scipy.stats as stats
import pandas as pd
from tqdm import tqdm

# 기본 경로
base_folder = r'C:\2025.01.Drought\2004'

# 사용할 누적 개월 수
window_sizes = [3,6, 9, 12]

def compute_spi_series(values):
    """1D 시계열(values)에 대해 감마분포 기반 SPI 계산"""
    arr = np.array(values)
    if np.all(np.isnan(arr)) or np.sum(arr > 0) < 30:
        return np.full_like(arr, np.nan)
    try:
        a, loc, scale = stats.gamma.fit(arr, floc=0)
        cdf = stats.gamma.cdf(arr, a, loc=loc, scale=scale)
        return stats.norm.ppf(cdf)
    except Exception:
        return np.full_like(arr, np.nan)

for window in window_sizes:
    # 입력·출력 폴더 설정
    in_folder  = os.path.join(base_folder, f'{window}mon_output_Cokriging')
    out_folder = os.path.join(base_folder, f'SPI{window}(Cok)')
    os.makedirs(out_folder, exist_ok=True)

    # 파일 모으기
    file_list = sorted([
        f for f in os.listdir(in_folder)
        if f.endswith('.nc') and f'{window}mon' in f
    ])
    dates = [pd.to_datetime(f.split('_')[0], format='%Y%m') for f in file_list]

    # xarray Dataset 리스트 생성
    datasets = []
    for fname, date in zip(file_list, dates):
        ds = xr.open_dataset(os.path.join(in_folder, fname))
        ds = ds.expand_dims(time=[np.datetime64(date, 'ns')])
        datasets.append(ds)

    # 병합
    full = xr.concat(datasets, dim='time')
    precip = full['precipitation']  # (time, lat, lon)

    # SPI 계산을 위한 배열 초기화
    nt, ny, nx = precip.shape
    spi_arr = np.full((nt, ny, nx), np.nan)

    for i in tqdm(range(ny), desc=f'SPI{window} 계산 (위도)'):
        for j in range(nx):
            spi_arr[:, i, j] = compute_spi_series(precip[:, i, j].values)

    # xarray Dataset으로 재구성
    spi_ds = xr.Dataset(
        {f'SPI{window}': (['time', 'lat', 'lon'], spi_arr)},
        coords={
            'time': full.time,
            'latitude': full.latitude,
            'longitude': full.longitude
        }
    )

    # 시간별로 분할 저장
    for idx, ts in enumerate(spi_ds.time.values):
        ym = pd.to_datetime(str(ts)).strftime('%Y%m')
        single = (
            spi_ds.isel(time=idx)
                  .drop_vars('time')
                  .expand_dims(time=[ts])
        )
        out_file = os.path.join(out_folder, f"{ym}_SPI{window}.nc")
        single.to_netcdf(out_file)
        print(f"✅ 저장됨: {out_file}")

    # 메모리 해제
    full.close()
    spi_ds.close()
    for ds in datasets:
        ds.close()


SPI3 계산 (위도): 100%|██████████| 101/101 [00:00<00:00, 109.23it/s]


✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200403_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200404_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200405_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200406_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200407_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200408_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200409_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200410_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200411_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200412_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200501_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200502_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200503_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200504_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200505_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200506_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200507_SPI3.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI3(Cok)\200508_

SPI6 계산 (위도): 100%|██████████| 101/101 [00:00<00:00, 111.82it/s]


✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200406_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200407_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200408_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200409_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200410_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200411_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200412_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200501_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200502_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200503_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200504_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200505_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200506_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200507_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200508_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200509_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200510_SPI6.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI6(Cok)\200511_

SPI9 계산 (위도): 100%|██████████| 101/101 [00:00<00:00, 109.41it/s]


✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200409_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200410_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200411_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200412_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200501_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200502_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200503_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200504_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200505_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200506_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200507_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200508_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200509_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200510_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200511_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200512_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200601_SPI9.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI9(Cok)\200602_

SPI12 계산 (위도): 100%|██████████| 101/101 [00:00<00:00, 105.20it/s]


✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200412_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200501_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200502_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200503_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200504_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200505_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200506_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200507_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200508_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200509_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200510_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200511_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200512_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200601_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200602_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200603_SPI12.nc
✅ 저장됨: C:\2025.01.Drought\2004\SPI12(Cok)\200604_SPI12.nc
✅ 저장됨: C:\2025

In [ ]:
# ── 0.  (처음 한 번만) 필요한 패키지 ────────────────────────────────
#%pip install -q xarray netCDF4 rioxarray cf_xarray rasterio

import xarray as xr
import rioxarray                               # .rio 기능 추가
from pathlib import Path

# ── 1.  파일 경로 ---------------------------------------------------------
ref_path = Path(r"C:\Yeonji\2025.01.Drought\2004\SPI(Cok)\SPI3(Cok)/200404_SPI3.nc")           # 클리핑 기준
src_path = Path(r"C:\Yeonji\2025.01.Drought\2004\20250615_181422\downscaled_results_2004_2023/20040101_Downscaled_Precipitation.nc")  # 자를 파일
out_path = Path(r"C:\Yeonji\2025.01.Drought\2004/clip/20040101_Downscaled_Precipitation_clip.nc")

ref = xr.open_dataset(ref_path)
src = xr.open_dataset(src_path)

# ── 1) 좌표계(CRS) 태그 확실히 지정 ──────────────────────────────────
for ds in (ref, src):
    ds.rio.write_crs("EPSG:4326", inplace=True)      # 이미 있으면 자동 무시

# ── 2) ‘lon/lat’을 rioxarray 가 인식하도록 공간 차원으로 지정 ────────
#   → 차원 이름이 다르면 바꿔 주세요 (예: longitude/latitude)
for name, ds in zip(('ref', 'src'), (ref, src)):
    # If lon/lat are dimensions but missing coordinates, assign them from longitude/latitude
    if set(("lon", "lat")).issubset(ds.dims):
        # Assign coordinates if missing
        if "lon" not in ds.coords:
            if "longitude" in ds.coords:
                ds = ds.assign_coords(lon=ds.longitude)
            elif "longitude" in ds.variables:
                ds = ds.assign_coords(lon=ds["longitude"].values)
            else:
                ds = ds.assign_coords(lon=ds['lon'])
        if "lat" not in ds.coords:
            if "latitude" in ds.coords:
                ds = ds.assign_coords(lat=ds.latitude)
            elif "latitude" in ds.variables:
                ds = ds.assign_coords(lat=ds["latitude"].values)
            else:
                ds = ds.assign_coords(lat=ds['lat'])
        ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    elif set(("longitude", "latitude")).issubset(ds.dims):
        ds = ds.rename({"longitude": "lon", "latitude": "lat"})
        ds = ds.assign_coords(lon=ds.lon, lat=ds.lat)
        ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    else:
        raise ValueError("경위도 차원 이름을 찾을 수 없습니다. NetCDF 구조를 확인하세요.")
    # Update the reference in the outer scope
    if name == 'ref':
        ref = ds
    else:
        src = ds

# Ensure src's lat/lon coordinates are float and match the bounds
if "lat" in src.coords and "lon" in src.coords:
    src = src.assign_coords(lat=src.lat.astype(float), lon=src.lon.astype(float))

# ── 3) 기준 파일의 경계 상자 가져오기 ───────────────────────────────
xmin, ymin, xmax, ymax = ref.rio.bounds()

# 진단: 좌표 범위 출력
print("ref bounds:", xmin, ymin, xmax, ymax)
print("src lon range:", float(src.lon.min()), float(src.lon.max()))
print("src lat range:", float(src.lat.min()), float(src.lat.max()))

# 만약 src의 lat/lon 방향이 ref와 다르면 뒤집기
if float(src.lat[0]) > float(src.lat[-1]):
    src = src.sortby('lat')
if float(src.lon[0]) > float(src.lon[-1]):
    src = src.sortby('lon')

# 다시 범위 출력
print("src (sorted) lon range:", float(src.lon.min()), float(src.lon.max()))
print("src (sorted) lat range:", float(src.lat.min()), float(src.lat.max()))

# ── 4) clip_box 실행 ───────────────────────────────────────────────
try:
    src_clip = src.rio.clip_box(xmin, ymin, xmax, ymax)
except Exception as e:
    print("❌ 클리핑 실패:", e)
    raise

# ── 5) 저장 ────────────────────────────────────────────────────────
encoding = {v: {"zlib": True, "complevel": 4} for v in src_clip.data_vars}
src_clip.to_netcdf(out_path, encoding=encoding)

print("✅ 클리핑 완료 →", out_path)


ref bounds: 126.1 33.05 127.1 33.65
src lon range: 0.0 100.0
src lat range: 0.0 60.0
src (sorted) lon range: 0.0 100.0
src (sorted) lat range: 0.0 60.0
❌ 클리핑 실패: x dimension not found. 'rio.set_spatial_dims()' or using 'rename()' to change the dimension name to 'x' can address this. Data variable: precipitation


MissingSpatialDimensionError: x dimension not found. 'rio.set_spatial_dims()' or using 'rename()' to change the dimension name to 'x' can address this. Data variable: precipitation